# GET 324 — Laboratory Exercise 10 (Mini-Project)
## Cloud Computing and AI Model Deployment for Engineering Applications
### Task: Binary Image Classification — **Deer vs Antelope**

**Group Members:** _(fill in names & registration numbers)_
1. Name — Reg No — GitHub username
2. Name — Reg No — GitHub username
3. Name — Reg No — GitHub username

**Pipeline covered in this notebook (CLO5, CLO7, CLO8):**
1. Environment setup
2. Real-world dataset acquisition (Kaggle — **no synthetic data**)
3. Dataset preparation & preprocessing
4. CNN model design (transfer learning — MobileNetV2)
5. Model training
6. Model evaluation (accuracy, precision, recall, F1, confusion matrix, ROC-AUC)
7. Saving the trained model (`.h5` / `.keras`)
8. Generating `app.py` for Streamlit deployment
9. Instructions for GitHub + Streamlit Cloud deployment

> Run this notebook top-to-bottom in **Google Colab** with a GPU runtime:
> `Runtime → Change runtime type → T4 GPU`


## 1. Environment Setup

In [ ]:
# 1.1 Install/confirm required libraries (Colab has most pre-installed)
!pip install -q kaggle tensorflow scikit-learn seaborn matplotlib pillow streamlit
print("Environment ready.")

In [ ]:
import os, shutil, random, pathlib, json, zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Dataset Acquisition (Real Images — No Synthetic Data)

We use the public Kaggle dataset **"Animal Image Dataset — 90 Different Animals"**
(by Sourav Banerjee), which contains real, hand-curated photographs and includes
native `deer` and `antelope` classes. We filter the dataset down to only those
two classes for this binary classification task.

Dataset reference: https://www.kaggle.com/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals

**To run this cell you need a Kaggle API token:**
1. Go to https://www.kaggle.com/settings → "Create New Token" → downloads `kaggle.json`
2. Upload it when prompted below.


In [ ]:
from google.colab import files
print("Please upload your kaggle.json API token file:")
uploaded = files.upload()  # upload kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials configured.")

In [ ]:
# Download the real-world dataset directly from Kaggle
!kaggle datasets download -d iamsouravbanerjee/animal-image-dataset-90-different-animals -p /content/raw --unzip
print("Download complete.")
!find /content/raw -maxdepth 2 -type d | head -20

In [ ]:
# 2.1 Locate the two classes we need and copy them into a clean binary-task folder
RAW_ROOT = pathlib.Path('/content/raw')

# The dataset ships with a top folder typically named "animals/animals"
candidate_dirs = [p for p in RAW_ROOT.rglob('*') if p.is_dir() and p.name.lower() in ('deer', 'antelope')]
assert len(candidate_dirs) >= 2, f"Could not locate both class folders automatically: {candidate_dirs}"

class_dirs = {}
for p in candidate_dirs:
    class_dirs[p.name.lower()] = p
print("Found class folders:")
for k, v in class_dirs.items():
    n = len(list(v.glob('*')))
    print(f"  {k}: {v}  ({n} images)")

In [ ]:
# 2.2 Build a clean project dataset directory: /content/data/{deer,antelope}
DATA_ROOT = pathlib.Path('/content/data')
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)

for cls_name, src_dir in class_dirs.items():
    dest_dir = DATA_ROOT / cls_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    for i, img_path in enumerate(sorted(src_dir.glob('*'))):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue
        shutil.copy(img_path, dest_dir / f"{cls_name}_{i:04d}{img_path.suffix.lower()}")

for cls_name in class_dirs:
    n = len(list((DATA_ROOT / cls_name).glob('*')))
    print(f"{cls_name}: {n} real images copied")

In [ ]:
# 2.3 Quick visual sanity check of the real dataset
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, cls_name in enumerate(['deer', 'antelope']):
    sample_imgs = random.sample(list((DATA_ROOT / cls_name).glob('*')), 5)
    for col, img_path in enumerate(sample_imgs):
        img = tf.keras.utils.load_img(img_path)
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls_name)
        axes[row, col].axis('off')
plt.suptitle("Sample real images from dataset (deer vs antelope)")
plt.tight_layout()
plt.show()

## 3. Dataset Preparation & Preprocessing

We split the real images into train (70%), validation (15%), and test (15%) sets,
then build `tf.data` pipelines with resizing, normalization, and augmentation.


In [ ]:
SPLIT_ROOT = pathlib.Path('/content/split')
if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

splits = {'train': 0.70, 'val': 0.15, 'test': 0.15}
for cls_name in ['deer', 'antelope']:
    imgs = list((DATA_ROOT / cls_name).glob('*'))
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * splits['train'])
    n_val = int(n * splits['val'])

    parts = {
        'train': imgs[:n_train],
        'val': imgs[n_train:n_train + n_val],
        'test': imgs[n_train + n_val:]
    }
    for split_name, files_ in parts.items():
        out_dir = SPLIT_ROOT / split_name / cls_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for f in files_:
            shutil.copy(f, out_dir / f.name)

for split_name in splits:
    for cls_name in ['deer', 'antelope']:
        n = len(list((SPLIT_ROOT / split_name / cls_name).glob('*')))
        print(f"{split_name}/{cls_name}: {n} images")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_ROOT / 'train', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_ROOT / 'val', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', seed=SEED
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_ROOT / 'test', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', seed=SEED, shuffle=False
)

class_names = train_ds.class_names  # alphabetical: ['antelope', 'deer']
print("Class order (label 0 / 1):", class_names)

with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

In [ ]:
# Performance: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

# Data augmentation layer (applied only during training, inside the model)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='data_augmentation')

## 4. CNN Model Design — Transfer Learning (MobileNetV2)

We use **MobileNetV2** pretrained on ImageNet as a frozen feature extractor,
followed by a custom classification head, then fine-tune the top layers.
This gives strong accuracy on a modest real-image dataset while staying
lightweight enough for cloud deployment.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze for initial training

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)  # binary classification

model = tf.keras.Model(inputs, outputs, name='deer_vs_antelope_cnn')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
model.summary()

## 5. Model Training

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
checkpoint = callbacks.ModelCheckpoint('best_model_stage1.keras', monitor='val_accuracy', save_best_only=True)

EPOCHS_STAGE1 = 15
history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
# 5.1 Fine-tuning: unfreeze the top layers of the base model for a few epochs
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30  # unfreeze last 30 layers only
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # low LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

EPOCHS_STAGE2 = 10
early_stop2 = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
checkpoint2 = callbacks.ModelCheckpoint('best_model_finetuned.keras', monitor='val_accuracy', save_best_only=True)

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=[early_stop2, checkpoint2]
)

In [ ]:
# 5.2 Plot training curves (combining both stages)
def combine_history(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(combine_history(history_1, history_2, 'accuracy'), label='train acc')
axes[0].plot(combine_history(history_1, history_2, 'val_accuracy'), label='val acc')
axes[0].axvline(x=len(history_1.history['accuracy'])-1, color='gray', linestyle='--', label='fine-tune start')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(combine_history(history_1, history_2, 'loss'), label='train loss')
axes[1].plot(combine_history(history_1, history_2, 'val_loss'), label='val loss')
axes[1].axvline(x=len(history_1.history['loss'])-1, color='gray', linestyle='--', label='fine-tune start')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Model Evaluation

In [ ]:
test_loss, test_acc, test_auc, test_prec, test_recall = model.evaluate(test_ds)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test AUC      : {test_auc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall   : {test_recall:.4f}")

In [ ]:
# 6.1 Collect predictions on the test set
y_true, y_prob = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0).flatten()
    y_true.extend(labels.numpy().flatten().tolist())
    y_prob.extend(preds.tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# 6.2 Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Deer vs Antelope')
plt.show()

In [ ]:
# 6.3 ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend(loc='lower right')
plt.show()

In [ ]:
# 6.4 Visual inspection of predictions on test images
plt.figure(figsize=(15, 8))
for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0).flatten()
    for i in range(min(10, len(images))):
        ax = plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        true_label = class_names[int(labels[i].numpy()[0])]
        pred_label = class_names[int(preds[i] >= 0.5)]
        color = 'green' if true_label == pred_label else 'red'
        plt.title(f"True: {true_label}\nPred: {pred_label} ({preds[i]:.2f})", color=color, fontsize=9)
        plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Save the Trained Model

We save the final model in Keras' native `.keras` format (recommended) — this is
the file the Streamlit app loads at inference time (see Laboratory Exercises 7 & 8).


In [ ]:
FINAL_MODEL_PATH = 'deer_antelope_model.keras'
model.save(FINAL_MODEL_PATH)
print(f"Model saved to {FINAL_MODEL_PATH}")

# Download to your machine (you'll upload this into your GitHub repo)
from google.colab import files
files.download(FINAL_MODEL_PATH)
files.download('class_names.json')

## 8. Streamlit Deployment App

The cell below **writes `app.py` to disk**. This is the exact file you will
push to GitHub and deploy on **Streamlit Community Cloud**
(`https://share.streamlit.io`). It loads `deer_antelope_model.keras`,
lets a user upload an image, and returns the predicted class with confidence.


In [ ]:
app_code = '''
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import json
import os

st.set_page_config(page_title="Deer vs Antelope Classifier", page_icon="\U0001F98C", layout="centered")

MODEL_PATH = "deer_antelope_model.keras"
CLASS_NAMES_PATH = "class_names.json"
IMG_SIZE = (224, 224)

@st.cache_resource
def load_model():
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(CLASS_NAMES_PATH, "r") as f:
        class_names = json.load(f)
    return model, class_names

def preprocess_image(image: Image.Image):
    image = image.convert("RGB").resize(IMG_SIZE)
    arr = tf.keras.utils.img_to_array(image)
    arr = np.expand_dims(arr, axis=0)
    return arr

def main():
    st.title("\U0001F98C Deer vs Antelope Classifier")
    st.write(
        "Upload an image and this CNN (MobileNetV2 transfer-learning model) "
        "will predict whether it shows a **deer** or an **antelope**."
    )
    st.caption("GET 324 — Cloud Computing and AI Model Deployment for Engineering Applications")

    if not os.path.exists(MODEL_PATH):
        st.error(f"Model file `{MODEL_PATH}` not found. Place it in the same folder as this app.")
        return

    model, class_names = load_model()

    uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        image = Image.open(uploaded_file)
        st.image(image, caption="Uploaded image", use_container_width=True)

        with st.spinner("Classifying..."):
            arr = preprocess_image(image)
            prob = float(model.predict(arr, verbose=0).flatten()[0])
            pred_idx = int(prob >= 0.5)
            pred_label = class_names[pred_idx]
            confidence = prob if pred_idx == 1 else 1 - prob

        st.subheader("Prediction")
        st.success(f"**{pred_label.upper()}**  (confidence: {confidence*100:.2f}%)")

        st.write("Class probabilities:")
        st.progress(float(prob))
        st.write(f"- {class_names[0]}: {(1-prob)*100:.2f}%")
        st.write(f"- {class_names[1]}: {prob*100:.2f}%")

    st.markdown("---")
    st.caption("Model: MobileNetV2 transfer learning, fine-tuned on real Kaggle image data.")

if __name__ == "__main__":
    main()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py written to disk.")
print(app_code[:300], "...")

In [ ]:
requirements_txt = '''streamlit>=1.32
tensorflow>=2.15
numpy
pillow
'''
with open('requirements.txt', 'w') as f:
    f.write(requirements_txt)
print(requirements_txt)

In [ ]:
# Download the deployment files to your machine as well
from google.colab import files
files.download('app.py')
files.download('requirements.txt')

## 9. Deployment Instructions (GitHub + Streamlit Community Cloud)

1. **Create a GitHub repository** (e.g. `deer-vs-antelope-classifier`) and add these files
   at the repo root:
   - `app.py`
   - `requirements.txt`
   - `deer_antelope_model.keras`
   - `class_names.json`
   - `README.md`
   - `report.md`

2. **Push to GitHub:**
   ```bash
   git init
   git add app.py requirements.txt deer_antelope_model.keras class_names.json README.md report.md
   git commit -m "Deer vs Antelope CNN classifier - GET 324 mini-project"
   git branch -M main
   git remote add origin https://github.com/<your-username>/deer-vs-antelope-classifier.git
   git push -u origin main
   ```

3. **Deploy on Streamlit Community Cloud:**
   - Go to https://share.streamlit.io and sign in with GitHub.
   - Click **"New app"**, select your repository, branch `main`, and file `app.py`.
   - Click **Deploy**. Streamlit will install `requirements.txt` and launch the app.
   - You will get a public URL like `https://<your-app-name>.streamlit.app`.

4. **Test the deployed app** by uploading a few real deer/antelope photos and
   confirm the predictions match your local evaluation results.

5. Submit: `app.py`, GitHub repo link, deployed app URL, this notebook, the
   report, and group member details — as required in the assignment brief.


## 10. Brief Report (see `report.md`)

A ready-to-submit 100–150 word report (dataset source, usage instructions,
challenges, and improvements) has been prepared separately as `report.md`.
